# MobileNetV4 temporal cursor-hotspot detector

Select **Runtime → Change runtime type → GPU**. Create a fresh upload locally with `bash training/create_colab_bundle.sh`, then upload `yolo-cursor-colab.zip`. No generated frames are stored.

In [ ]:
import subprocess, torch
subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.is_available(), 'Enable a GPU runtime and reconnect.'
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name}; VRAM: {props.total_memory / 1024**3:.1f} GiB')

In [ ]:
%pip install -q timm==1.0.19 polars==1.35.2 pyarrow onnx

In [ ]:
from google.colab import files
from pathlib import Path
import os, shutil, zipfile
uploaded = files.upload()
bundle = next((Path(name) for name in uploaded if name.endswith('.zip')), None)
assert bundle, 'Upload yolo-cursor-colab.zip.'
workspace = Path('/content/yolo-cursor')
if workspace.exists(): shutil.rmtree(workspace)
with zipfile.ZipFile(bundle) as archive: archive.extractall('/content')
os.chdir(workspace)
print('Workspace:', Path.cwd())

In [ ]:
# Inspect an actual runtime sequence representation and target.
import matplotlib.pyplot as plt
from training.temporal_dataset import TemporalCursorDataset
sample = TemporalCursorDataset('train', length=1, dynamic=False)[0]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for axis, image, title in zip(axes, [*sample['input'], sample['heatmap'][0]], ['current grayscale', '|t - t-1|', '|t - t-2|', 'target heatmap']):
    axis.imshow(image, cmap='gray', vmin=0, vmax=1); axis.set_title(title); axis.axis('off')
print('input', tuple(sample['input'].shape), 'heatmap', tuple(sample['heatmap'].shape), 'hotspot', sample['position'].tolist(), 'type', sample['cursor_type'])

In [ ]:
# Train one stage end-to-end; pretrained grayscale features adapt to motion channels.
vram_gib = props.total_memory / 1024**3
batch = 64 if vram_gib >= 22 else 32
os.environ.update({
    'CURSOR_DEVICE': 'cuda', 'CURSOR_BATCH': str(batch), 'CURSOR_WORKERS': '2',
    'CURSOR_TRAIN_SAMPLES': '10000', 'CURSOR_VAL_SAMPLES': '1000',
    'CURSOR_EPOCHS': '20', 'CURSOR_LR': '0.001',
})
print(f'Using batch {batch}')
%run training/train_temporal.py

In [ ]:
# Export even when training was stopped early, then download both files.
from training.temporal_model import TemporalCursorExport, TemporalCursorNet
output = Path('training/runs/mobilenetv4-temporal')
best = output / 'best.pt'
onnx_path = output / 'mobilenetv4-temporal.onnx'
assert best.is_file(), f'Missing checkpoint: {best}'
checkpoint = torch.load(best, map_location='cpu', weights_only=False)
model = TemporalCursorNet(pretrained=False)
model.load_state_dict(checkpoint['model'])
model.eval()
torch.onnx.export(
    TemporalCursorExport(model), torch.zeros(1, 3, 640, 640), onnx_path,
    input_names=['frames'], output_names=['heatmap', 'offset'],
    opset_version=17, dynamo=False,
)
print('Exported', onnx_path, 'from epoch', checkpoint['epoch'], checkpoint['metrics'])
files.download(str(onnx_path))
files.download(str(best))